# Generation, oversampling, and traceability

Generate synthetic rows and inspect their trace records.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src" / "mimic").exists() else Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from mimic import MIMIC, GenerationPolicy, ResNetEncoder, ForestConditionalSampler

rng = np.random.default_rng(2)

def make_spiral(label, n, phase, noise=0.12):
    theta = np.linspace(0.45, 4.8 * np.pi, n)
    radius = np.linspace(0.25, 4.0, n)
    x = radius * np.cos(theta + phase) + rng.normal(0, noise, n)
    y = radius * np.sin(theta + phase) + rng.normal(0, noise, n)
    return pd.DataFrame({"x": x, "y": y, "label": label})

major = make_spiral("majority", 180, phase=0.0)
minor_full = make_spiral("minority", 180, phase=np.pi)
minor = minor_full.sample(n=90, random_state=2).sort_index().reset_index(drop=True)

df = pd.concat([major, minor], ignore_index=True)
df["id"] = np.arange(len(df))

display(df["label"].value_counts().rename_axis("label").to_frame("count").style.set_caption("Original class balance"))
display(df.head(8).style.set_caption("Original two-spiral training rows"))


,count
label,
majority,180
minority,90


,x,y,label,id
0,0.247798,0.205585,majority,0
1,0.170810,0.004947,majority,1
2,0.189107,0.061862,majority,2
3,-0.052729,0.212612,majority,3
4,0.453989,0.205364,majority,4
5,0.369114,0.451555,majority,5
6,0.182417,0.392399,majority,6
7,0.299736,0.304533,majority,7


In [2]:
mimic = MIMIC(
    ignore_columns=["id"],
    regression_columns=["x", "y"],
    classification_columns=["label"],
    encoder=ResNetEncoder(embedding_dim=8, hidden_dim=8, n_layers=2, dropout=0.05, max_epochs=30, patience=5, batch_size=64, random_state=2),
    decoder=ForestConditionalSampler(n_estimators=120, random_state=2),
    policy=GenerationPolicy(method="displacement", neighbour_mode="mutual", n_neighbors=5, lambda_range=(0.25, 0.75)),
    n_bootstrap=1,
    random_state=2,
)
mimic.fit(df)
synthetic, trace = mimic.sample(12, return_trace=True)

embedding_trace = trace.loc[trace["trace_type"].eq("embedding")].set_index("sample_index")
generated_trace = synthetic.join(
    embedding_trace[
        [
            "anchor_index",
            "displacement_from_index",
            "displacement_to_index",
            "lambda",
            "neighbour_mode",
            "decoder",
        ]
    ],
)
cell_trace = trace.loc[trace["trace_type"].eq("cell")].dropna(axis=1, how="all")

display(generated_trace.head(8).style.set_caption("Generated rows with embedding trace"))
display(cell_trace.head(12).style.set_caption("Per-cell stochastic sampling trace"))


,x,y,label,anchor_index,displacement_from_index,displacement_to_index,lambda,neighbour_mode,decoder
0,2.435643,-0.150861,minority,226.000000,225.000000,226.000000,0.399246,mutual,ForestConditionalSampler
1,-0.600538,-2.301636,minority,219.000000,220.000000,219.000000,0.550050,mutual,ForestConditionalSampler
2,-0.357606,-1.582583,minority,219.000000,264.000000,219.000000,0.343951,mutual,ForestConditionalSampler
3,0.158147,0.537635,minority,237.000000,236.000000,237.000000,0.387485,mutual,ForestConditionalSampler
4,-2.480947,2.792487,majority,177.000000,179.000000,177.000000,0.325031,mutual,ForestConditionalSampler
5,0.900847,0.589895,minority,202.000000,203.000000,202.000000,0.584649,mutual,ForestConditionalSampler
6,-2.512407,-0.367271,minority,114.000000,115.000000,112.000000,0.733718,mutual,ForestConditionalSampler
7,1.460423,2.595876,minority,234.000000,232.000000,233.000000,0.445812,mutual,ForestConditionalSampler


,trace_type,sample_index,method,decoder,sweep,column,task,sampled_value,conditioning,embedding_slice_start,embedding_slice_stop,member_index,source_position,source_index,source_weight,leaf_support_size,sampled_class_code,class_probabilities
12,cell,0,gibbs_forest,ForestConditionalSampler,0.000000,x,regression,2.435643,z_minus_j,0.000000,8.000000,0.000000,226.000000,226.000000,0.578889,19.000000,nan,nan
13,cell,1,gibbs_forest,ForestConditionalSampler,0.000000,x,regression,-0.603343,z_minus_j,0.000000,8.000000,0.000000,264.000000,264.000000,0.202778,16.000000,nan,nan
14,cell,2,gibbs_forest,ForestConditionalSampler,0.000000,x,regression,-0.228126,z_minus_j,0.000000,8.000000,0.000000,187.000000,187.000000,0.156190,21.000000,nan,nan
15,cell,3,gibbs_forest,ForestConditionalSampler,0.000000,x,regression,0.158147,z_minus_j,0.000000,8.000000,0.000000,237.000000,237.000000,0.271885,24.000000,nan,nan
16,cell,4,gibbs_forest,ForestConditionalSampler,0.000000,x,regression,-2.460587,z_minus_j,0.000000,8.000000,0.000000,116.000000,116.000000,0.194861,15.000000,nan,nan
17,cell,5,gibbs_forest,ForestConditionalSampler,0.000000,x,regression,0.900847,z_minus_j,0.000000,8.000000,0.000000,199.000000,199.000000,0.373750,26.000000,nan,nan
18,cell,6,gibbs_forest,ForestConditionalSampler,0.000000,x,regression,-2.512407,z_minus_j,0.000000,8.000000,0.000000,112.000000,112.000000,0.146052,15.000000,nan,nan
19,cell,7,gibbs_forest,ForestConditionalSampler,0.000000,x,regression,1.460423,z_minus_j,0.000000,8.000000,0.000000,234.000000,234.000000,0.348889,21.000000,nan,nan
20,cell,8,gibbs_forest,ForestConditionalSampler,0.000000,x,regression,-0.078141,z_minus_j,0.000000,8.000000,0.000000,129.000000,129.000000,0.118912,16.000000,nan,nan
21,cell,9,gibbs_forest,ForestConditionalSampler,0.000000,x,regression,0.148406,z_minus_j,0.000000,8.000000,0.000000,189.000000,189.000000,0.241389,21.000000,nan,nan


In [ ]:
minority_needed = df["label"].value_counts()["majority"] - df["label"].value_counts()["minority"]
minority_synthetic, minority_trace = mimic.sample(
    minority_needed,
    condition={"label": "minority"},
    return_trace=True,
)
balanced = pd.concat([df.drop(columns=["id"]), minority_synthetic], ignore_index=True)

display(balanced["label"].value_counts().rename_axis("label").to_frame("count").style.set_caption("Class balance after minority-conditioned displacement oversampling"))
minority_embedding_trace = minority_trace.loc[minority_trace["trace_type"].eq("embedding")].set_index("sample_index")
minority_generated_trace = minority_synthetic.join(
    minority_embedding_trace[
        [
            "anchor_index",
            "displacement_from_index",
            "displacement_to_index",
            "lambda",
            "neighbour_mode",
            "condition",
            "decoder",
        ]
    ],
)
minority_cell_trace = minority_trace.loc[minority_trace["trace_type"].eq("cell")].dropna(axis=1, how="all")

display(minority_generated_trace.head(8).style.set_caption("Synthetic minority rows with embedding trace"))
display(minority_cell_trace.head(12).style.set_caption("Minority per-cell stochastic sampling trace"))


In [ ]:
plot_df = pd.concat(
    [
        df.drop(columns=["id"]).assign(source="original"),
        minority_synthetic.assign(source="generated"),
    ],
    ignore_index=True,
)

fig, ax = plt.subplots(figsize=(6, 5))
for (source, label), part in plot_df.groupby(["source", "label"]):
    if source == "original" and label == "majority":
        color, marker, alpha, size = "#9aa0a6", "o", 0.42, 28
    elif source == "original" and label == "minority":
        color, marker, alpha, size = "#1f77b4", "o", 0.9, 46
    else:
        color, marker, alpha, size = "#ff2e0e", "o", 0.9, 52
    ax.scatter(part["x"], part["y"], label=f"{source} {label}", color=color, marker=marker, alpha=alpha, s=size)
ax.set_title("Displacement oversampling on undersampled two spirals")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_aspect("equal", adjustable="box")
ax.legend(frameon=False)
fig.tight_layout()


## Identity-space configurable baseline

Set the full MIMIC estimator here, using `IdentityEncoder` and `IdentityDecoder` to generate in the preprocessed original feature space. Change `identity_policy` to switch between SMOTE-style interpolation and displacement.

In [ ]:
import sys
from pathlib import Path

from mimic import IdentityDecoder, IdentityEncoder

notebook_dir = Path.cwd() / "notebooks" if (Path.cwd() / "notebooks").exists() else Path.cwd()
if str(notebook_dir) not in sys.path:
    sys.path.append(str(notebook_dir))

from mimic_notebook_utils import identity_generation_plot

identity_policy = GenerationPolicy(
    method="displacement",  # change to "displacement" for the displacement variant
    neighbour_mode="mutual",
    n_neighbors=5,
    lambda_range=(0.25, 0.75),
)

identity_mimic = MIMIC(
    ignore_columns=["id"],
    regression_columns=["x", "y"],
    classification_columns=["label"],
    encoder=IdentityEncoder(),
    decoder=IdentityDecoder(),
    policy=identity_policy,
    n_bootstrap=1,
    random_state=2,
)

identity_summary, identity_synthetic, identity_trace, identity_fig, identity_ax = identity_generation_plot(
    identity_mimic,
    df,
    n_samples=minority_needed,
    condition={"label": "minority"},
    title=f"Identity-space {identity_policy.method} oversampling",
)

display(identity_summary.style.set_caption("Identity-space generation summary"))
display(identity_synthetic.head(8).style.set_caption("Identity-space synthetic minority rows"))
display(identity_trace.head(8).style.set_caption("Identity-space generation trace"))